In [1]:
# spark.stop()

In [2]:
import os
from pyspark.sql import SparkSession, types as t, functions as F
from pyspark.sql.types import StringType, FloatType, IntegerType

# https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar

spark = (
    SparkSession
    .builder
    # .master("spark://spark-master:7077")
    .appName("Testing Transformations")
    .config("spark.jars", "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar") # GCS Connector
    .getOrCreate()
)

# Google Cloud Service Account Credentials
spark._jsc.hadoopConfiguration().set("google.cloud.auth.service.account.json.keyfile",os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"))

spark

your 131072x1 screen size is bogus. expect trouble
25/05/28 00:54:47 WARN Utils: Your hostname, Trydex resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/05/28 00:54:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/28 00:54:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/05/28 00:54:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
bucket='gs://zoomcamp-454219-ade-pipeline/data/pq/'
schema = 'drug'
year = 2025
event = "drug-event-part-1-of-34.parquet" # 2025
# year = 2004
# event = "drug-event-part-1-of-20.parquet" # 2004

df = (
    spark
    .read
    .parquet(bucket+f'{schema}/{year}/{event}')
    )
print(f"Count: {df.count()}")
# print(df.printSchema())
# patient.show()

Count: 106119


In [4]:
df.printSchema()

root
 |-- patientid: string (nullable = true)
 |-- actiondrug: string (nullable = true)
 |-- drugcharacterization: string (nullable = true)
 |-- medicinalproduct: string (nullable = true)
 |-- activesubstancename: string (nullable = true)
 |-- drugindication: string (nullable = true)
 |-- drugadministrationroute: string (nullable = true)
 |-- drugstartdate: string (nullable = true)
 |-- drugenddate: string (nullable = true)
 |-- drugdosagetext: string (nullable = true)
 |-- drugstructuredosagenumb: string (nullable = true)
 |-- drugstructuredosageunit: string (nullable = true)
 |-- drugtreatmentduration: string (nullable = true)
 |-- drugtreatmentdurationunit: string (nullable = true)
 |-- drugrecurreadministration: string (nullable = true)



In [5]:
from pyspark.sql import functions as F

def clean_date_column(df, col_name):
        """ Function to clean date column for spark"""

        # Fix missing days or months
        temp_df = df.withColumn(
            col_name,
            (
                F
                .when(F.length(col_name) == 4, F.concat(col_name,F.lit("0101")))
                .when(F.length(col_name) == 6, F.concat(col_name,F.lit("01")))
                .otherwise(F.col(col_name))
            )
        )

        # Set date constraints
        temp_df = temp_df.withColumn(
            col_name,
            (
                F
                .when(F.col(col_name) < F.lit("19000101"), None)
                .otherwise(F.col(col_name))
            )
        )

        # Cast to date
        temp_df = temp_df.withColumn(col_name,(F.to_date(col_name,"yyyyMMdd")))

        return temp_df

In [6]:
from pyspark.sql.types import StringType, FloatType, IntegerType
from pyspark.sql import functions as F

class Drug:
    drug_administration_route_map = {
        "001": "Auricular (otic)",
        "002": "Buccal",
        "003": "Cutaneous",
        "004": "Dental",
        "005": "Endocervical",
        "006": "Endosinusial",
        "007": "Endotracheal",
        "008": "Epidural",
        "009": "Extra-amniotic",
        "010": "Hemodialysis",
        "011": "Intra corpus cavernosum",
        "012": "Intra-amniotic",
        "013": "Intra-arterial",
        "014": "Intra-articular",
        "015": "Intra-uterine",
        "016": "Intracardiac",
        "017": "Intracavernous",
        "018": "Intracerebral",
        "019": "Intracervical",
        "020": "Intracisternal",
        "021": "Intracorneal",
        "022": "Intracoronary",
        "023": "Intradermal",
        "024": "Intradiscal (intraspinal)",
        "025": "Intrahepatic",
        "026": "Intralesional",
        "027": "Intralymphatic",
        "028": "Intramedullar (bone marrow)",
        "029": "Intrameningeal",
        "030": "Intramuscular",
        "031": "Intraocular",
        "032": "Intrapericardial",
        "033": "Intraperitoneal",
        "034": "Intrapleural",
        "035": "Intrasynovial",
        "036": "Intratumor",
        "037": "Intrathecal",
        "038": "Intrathoracic",
        "039": "Intratracheal",
        "040": "Intravenous bolus",
        "041": "Intravenous drip",
        "042": "Intravenous (not otherwise specified)",
        "043": "Intravesical",
        "044": "Iontophoresis",
        "045": "Nasal",
        "046": "Occlusive dressing technique",
        "047": "Ophthalmic",
        "048": "Oral",
        "049": "Oropharingeal",
        "050": "Other",
        "051": "Parenteral",
        "052": "Periarticular",
        "053": "Perineural",
        "054": "Rectal",
        "055": "Respiratory (inhalation)",
        "056": "Retrobulbar",
        "057": "Sunconjunctival",
        "058": "Subcutaneous",
        "059": "Subdermal",
        "060": "Sublingual",
        "061": "Topical",
        "062": "Transdermal",
        "063": "Transmammary",
        "064": "Transplacental",
        "065": "Unknown",
        "066": "Urethral",
        "067": "Vaginal"
    }

    def __init__(self,df):
        self.df = df

    def get_df(self):
        return self.df

    def cast(self):
        self.df = (
            self.df
            .withColumn("patientid", F.col("patientid").cast(StringType()))
            .withColumn("medicinalproduct", F.col("medicinalproduct").cast(StringType()))
            .withColumn("activesubstancename", F.col("activesubstancename").cast(StringType()))
            .withColumn("drugindication", F.col("drugindication").cast(StringType()))
            .withColumn("drugadministrationroute", F.col("drugadministrationroute").cast(StringType()))
            .withColumn("drugstartdate", F.col("drugstartdate").cast(StringType()))
            .withColumn("drugenddate", F.col("drugenddate").cast(StringType()))
            .withColumn("drugdosagetext", F.col("drugdosagetext").cast(StringType()))
            .withColumn("drugstructuredosagenumb", F.col("drugstructuredosagenumb").cast(FloatType()))
            .withColumn("drugstructuredosageunit", F.col("drugstructuredosageunit").cast(StringType()))
            .withColumn("drugtreatmentduration", F.col("drugtreatmentduration").cast(IntegerType()))
            .withColumn("drugtreatmentdurationunit", F.col("drugtreatmentdurationunit").cast(StringType()))
            .withColumn("drugrecurreadministration", F.col("drugrecurreadministration").cast(IntegerType()))
            .withColumn("actiondrug", F.col("actiondrug").cast(IntegerType()))
            .withColumn("drugcharacterization", F.col("drugcharacterization").cast(IntegerType()))
            )

    def transform(self):
        # Fix date
        self.df = clean_date_column(self.df, "drugstartdate").withColumnRenamed("drugstartdate", "drug_start_date")

        self.df = clean_date_column(self.df, "drugenddate").withColumnRenamed("drugenddate", "drug_end_date")

        map_expr = F.create_map([F.lit(i) for i in sum(self.drug_administration_route_map.items(),())])

        self.df = self.df.withColumn("drugadministrationroute", map_expr[F.col("drugadministrationroute")]).withColumnRenamed("drugadministrationroute", "administration_route")

        # Find and replace strings containing the word "Unknown"
        self.df = self.df.withColumn(
            "drugindication",
            F.when(
                F.col('drugindication').rlike("(?i)Unknown"),
                F.lit("Unknown")
            ).otherwise(F.col('drugindication'))
        ).withColumn(
            "drugindication",
            F.regexp_replace("drugindication",r"\^", "'")
        ).withColumnRenamed("drugindication","drug_indication")
        
        # Normalize dosage to mg
        self.df = self.df.withColumn(
            "dosage_mg",
            (
                F
                .when(F.col("drugstructuredosageunit") == "001", F.col("drugstructuredosagenumb") * 1e-6)
                .when(F.col("drugstructuredosageunit") == "002", F.col("drugstructuredosagenumb") * 1e-3)
                .when(F.col("drugstructuredosageunit") == "003", F.col("drugstructuredosagenumb") * 1)
                .when(F.col("drugstructuredosageunit") == "004", F.col("drugstructuredosagenumb") * 10**3)
                .otherwise(None)
            )
        ).drop("drugstructuredosageunit", "drugstructuredosagenumb")

        # Noramlized to days
        self.df = self.df.withColumn(
            "treatment_duration_days",
            (
                F
                .when(F.col("drugtreatmentdurationunit") == "801", F.col("drugtreatmentduration") * 365.25)
                .when(F.col("drugtreatmentdurationunit") == "802", F.col("drugtreatmentduration") * 30.46)
                .when(F.col("drugtreatmentdurationunit") == "803", F.col("drugtreatmentduration") * 7)
                .when(F.col("drugtreatmentdurationunit") == "804", F.col("drugtreatmentduration") * 1)
                .when(F.col("drugtreatmentdurationunit") == "805", F.col("drugtreatmentduration") / 24)
                .when(F.col("drugtreatmentdurationunit") == "806", F.col("drugtreatmentduration") / 1440)
                .otherwise(None)
            )
        ).drop("drugtreatmentdurationunit", "drugtreatmentduration")
        

        self.df = self.df.withColumn(
            "drug_reaction_after_readministration",
            (
                F
                .when(F.col("drugrecurreadministration") == 1, F.lit("Yes"))
                .when(F.col("drugrecurreadministration") == 2, F.lit("No"))
                .when(F.col("drugrecurreadministration") == 3, F.lit("Unknown"))
                .otherwise(None)
            )
        ).drop("drugrecurreadministration")
        

        self.df = self.df.withColumn(
            "actiondrug",
            (
                F
                .when(F.col("actiondrug") == 1, F.lit("Drug withdrawn"))
                .when(F.col("actiondrug") == 2, F.lit("Dose reduced"))
                .when(F.col("actiondrug") == 3, F.lit("Dose increased"))
                .when(F.col("actiondrug") == 4, F.lit("Dose not changed"))
                .when(F.col("actiondrug") == 5, F.lit("Unknown"))
                .when(F.col("actiondrug") == 6, F.lit("Not applicable"))
                .otherwise(None)
            )
        )

        self.df = self.df.withColumn(
            "drugcharacterization",
            (
                F
                .when(F.col("drugcharacterization") == 1, F.lit("Suspect"))
                .when(F.col("drugcharacterization") == 2, F.lit("Concomitant"))
                .when(F.col("drugcharacterization") == 3, F.lit("Interacting"))
                .otherwise(None)
            )
        )

        # print(self.df.columns)

        # # Handle null
        self.handle_null()

    def handle_null(self):
        fillna_dict = { 
            'actiondrug' : 'Unknown',
            'drugcharacterization': 'Unknown',
            'medicinalproduct': 'Unknown',
            'activesubstancename': 'Unknown',
            'drug_indication': 'Unknown',
            'administration_route': 'Unknown',
            # 'drug_start_date': '',
            # 'drug_end_date': '',
            'drugdosagetext': 'Not Specified',
            'dosage_mg': -1.0,
            'treatment_duration_days': -1,
            'drug_reaction_after_readministration': 'Unknown',

        }

        self.df = self.df.fillna(fillna_dict)

In [7]:
d = Drug(df)
d.cast()
d.transform()

In [8]:
df_processed = d.get_df()
df_processed.columns

['patientid',
 'actiondrug',
 'drugcharacterization',
 'medicinalproduct',
 'activesubstancename',
 'drug_indication',
 'administration_route',
 'drug_start_date',
 'drug_end_date',
 'drugdosagetext',
 'dosage_mg',
 'treatment_duration_days',
 'drug_reaction_after_readministration']

In [558]:
ddf = df_processed.toPandas()
ddf.shape

(106119, 13)

In [559]:
ddf.isna().sum()

patientid                                   0
actiondrug                                  0
drugcharacterization                        0
medicinalproduct                            0
activesubstancename                         0
drug_indication                             0
administration_route                        0
drug_start_date                         84371
drug_end_date                           96654
drugdosagetext                              0
dosage_mg                                   0
treatment_duration_days                     0
drug_reaction_after_readministration        0
dtype: int64

In [584]:
ddf.drug_start_date.sample(n=10)

38762    2024-06-04
2880           None
33041          None
68598    2008-01-01
18227    2023-12-14
87267          None
17823          None
90371    2020-10-06
43056          None
90703          None
Name: drug_start_date, dtype: object

In [481]:
ddf.drug_reaction_after_readministration.nunique()

3

In [499]:
test_df = ddf.loc[(ddf.drug_reaction_after_readministration.isna()), ['drug_reaction_after_readministration']].sample(n=10)
test_df

,drug_reaction_after_readministration
103235,None
57208,None
50362,None
63698,None
44940,None
27397,None
93235,None
54814,None
88084,None
17852,None


In [158]:
ddf[~ddf.drugindication.isna()].sample(n=20)

,patientid,actiondrug,drugcharacterization,medicinalproduct,activesubstancename,drugindication,administration_route,drug_start_date,drug_end_date,drugdosagetext,dosage_mg,treatment_duration_days,drug_reaction_after_readministration
62469,23d6f6e0-9157-41e7-ab0e-77fbf66994e2,None,Concomitant,KEPPRA,LEVETIRACETAM,Unknown,None,None,None,None,NaN,NaN,None
39805,3a6e8a29-fe6b-40fa-8583-2bf4790633c9,Unknown,Suspect,ARALAST NP,.ALPHA.1-PROTEINASE INHIBITOR HUMAN,Unknown,None,None,None,None,NaN,NaN,Unknown
13624,801c5691-4638-44a0-b6a8-87d9b28a99b6,None,Concomitant,LISINOPRIL,LISINOPRIL,Unknown,Unknown,None,None,None,NaN,NaN,None
2657,9446a4bb-96a8-4439-87c2-129e064ce88b,Unknown,Concomitant,ESTRADIOL VALERATE,ESTRADIOL VALERATE,Unknown,Unknown,None,None,None,NaN,NaN,None
74491,76911fd6-9aad-4414-8402-e8eb6d7d7aad,Unknown,Suspect,OMALIZUMAB,OMALIZUMAB,Asthma,Subcutaneous,None,None,None,225.0,NaN,None
61643,b3135764-5bc8-421f-b2cf-69b058c12ca5,None,Concomitant,ALLOPURINOL,ALLOPURINOL,Unknown,None,None,None,None,NaN,NaN,None
25795,e7869650-2c50-41bf-bab8-683ee3c20e02,Unknown,Suspect,BIMZELX,BIMEKIZUMAB-BKZX,Ankylosing spondylitis,Subcutaneous,2025-01-01,None,"160 MILLIGRAM, EV 4 WEEKS",160.0,NaN,None
59961,a367ea24-bd12-4a02-8c4c-6abe2c6965c9,Not applicable,Suspect,CLOPIDOGREL BISULFATE,CLOPIDOGREL BISULFATE,Antiplatelet therapy,Oral,2024-12-04,2024-12-31,"75 MG, QD",75.0,27.00,None
43061,4e89fcd7-e5a6-4c4d-8285-df3d73852c0a,None,Concomitant,TROSPIUM CHLORIDE,TROSPIUM CHLORIDE,Unknown,None,None,None,None,NaN,NaN,None
75161,cbf2bc45-1049-4055-b11a-bc0bacedaf68,Unknown,Suspect,PLEGRIDY,PEGINTERFERON BETA-1A,Multiple sclerosis,Other,2015-03-29,None,INJECT 125 MCG (1 INJECTOR) SUBCUTANEOUSLY EVE...,125000.0,NaN,Unknown
